In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

# --- GESTION DES CHEMINS ---
current_dir = os.getcwd()

# 1. Racine (pour 'import src...')
if current_dir not in sys.path:
    sys.path.append(current_dir)

# 2. Src (pour 'from config import...')
src_path = os.path.join(current_dir, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"📂 Racine : {current_dir}")
print(f"📂 Src    : {src_path}")

# --- IMPORTS ---
try:
    from config import Config
    print("✅ Config chargé.")
except ImportError:
    print("❌ Config introuvable (vérifiez src/config.py).")

# CORRECTION ICI : on importe depuis 'adr', pas 'models'
try:
    from src.models.adr import PI_DeepONet_ADR
    print("✅ Modèle chargé (src.models.adr).")
except ImportError as e:
    print(f"❌ Erreur import Modèle : {e}")

try:
    from src.training.smart_trainer import train_smart_time_marching
    from src.visualization.plots import (
        compare_model_vs_cn_snapshots, 
        animate_model_vs_cn, 
        plot_error_heatmaps, 
        evaluate_global_accuracy
    )
    print("✅ Trainer & Plots chargés.")
except ImportError as e:
    print(f"❌ Erreur import Trainer/Plots : {e}")

📂 Racine : /Users/emma.grospellier/Thèse/Projet_These_ADR
📂 Src    : /Users/emma.grospellier/Thèse/Projet_These_ADR/src
✅ Config chargé.
✅ Modèle chargé (src.models.adr).
✅ Trainer & Plots chargés.


In [2]:


# --- CELLULE 2 : FONCTION DE TEST ---
def run_smoke_test_notebook():
    print("🚀 DÉMARRAGE DU SMOKE TEST (Notebook Mode)...")

    # 1. OVERRIDE CONFIG (Version très légère pour le test)
    print("   🔧 Modification Config pour test rapide...")
    Config.n_warmup = 10           
    Config.n_iters_per_step = 20   
    Config.T_max = 0.2             
    Config.n_sample = 50           
    Config.batch_size = 16         
    Config.max_retry = 1
    # On sauvegarde dans un dossier spécifique
    Config.save_dir = "./test_results_notebook" 
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.backends.mps.is_available(): device = torch.device("mps")
    print(f"   📱 Device : {device}")

    # 2. INIT
    print("   🏗️ Initialisation modèle...")
    model = PI_DeepONet_ADR().to(device)

    # 3. TRAIN
    print("   🏋️ Entraînement rapide...")
    try:
        model = train_smart_time_marching(
            model, 
            bounds=Config.ranges, 
            n_warmup=Config.n_warmup, 
            n_iters_per_step=Config.n_iters_per_step
        )
    except Exception as e:
        print(f"❌ Erreur Training: {e}")
        return model # On retourne quand même pour inspecter

    # 4. PLOTS (Affichage direct)
    print("\n   🎨 Génération des plots...")
    os.makedirs(Config.save_dir, exist_ok=True)
    
    # On change le backend matplotlib temporairement pour afficher inline
    import matplotlib
    matplotlib.use('module://matplotlib_inline.backend_inline')
    
    try:
        # Snapshots
        compare_model_vs_cn_snapshots(model, save_dir=Config.save_dir)
        plt.show() # Force l'affichage
        print("   ✅ Snapshots générés.")
    except Exception as e: print(f"   ❌ Erreur Snapshots: {e}")

    try:
        # Heatmaps
        plot_error_heatmaps(model, Config.x_min, Config.x_max, Config.T_max, save_dir=Config.save_dir)
        plt.show()
        print("   ✅ Heatmaps générées.")
    except Exception as e: print(f"   ❌ Erreur Heatmaps: {e}")

    return model

# --- CELLULE 3 : LANCEMENT ---
model_trained = run_smoke_test_notebook()

🚀 DÉMARRAGE DU SMOKE TEST (Notebook Mode)...
   🔧 Modification Config pour test rapide...
   📱 Device : mps
   🏗️ Initialisation modèle...
   🏋️ Entraînement rapide...
⚡ DÉMARRAGE TRAINING (Config Driven)
   -> Warmup (t=0): 10 iters
   -> Time Step: 0.1, Max T: 0.2
   -> Batch Size: 16

🧊 PHASE 0 : Fixation Condition Initiale (t=0)...


Warmup IC: 100%|████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 18.70it/s]


   ✅ Warmup terminé.

⏳ --- PALIER TEMPOREL : [0, 0.1] ---


   ⚠️ Retry 1/1 (LR=5.0e-04, Err: 100.68%)...
   ❌ PALIER NON VALIDÉ (Err: 100.45%). Expansion forcée.

⏳ --- PALIER TEMPOREL : [0, 0.2] ---


   ⚠️ Retry 1/1 (LR=5.0e-04, Err: 99.79%)...
   ❌ PALIER NON VALIDÉ (Err: 105.79%). Expansion forcée.

🎉 Entraînement terminé.
Launching the statistical audit on 200 cases


100%|█████████████████████████████████████████████████████████████████████████████████| 200/200 [00:07<00:00, 28.29it/s]


Audit saved : ./test_results_notebook/global_audit_results.csv
Mean error L2: 107.74%

   🎨 Génération des plots...
✅ Figure sauvegardée : ./test_results_notebook/comparison_snapshots.png
   ✅ Snapshots générés.
✅ Heatmaps sauvegardées : ./test_results_notebook/error_heatmaps.png
   ✅ Heatmaps générées.


In [3]:
%%writefile launch.slurm
#!/bin/bash
#SBATCH --job-name=DeepONet_ADR     # Nom du job
#SBATCH --output=slurm/log/%x_%j.out
#SBATCH --error=slurm/log/%x_%j.err
#SBATCH -A fdb@v100                 # <--- TON COMPTE PROJET ICI
#SBATCH --constraint=v100-32g
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=10
#SBATCH --hint=nomultithread
#SBATCH --time=20:00:00
#SBATCH --qos=qos_gpu-t3

# Nettoyage
module purge
module load pytorch-gpu/py3/2.1.1

# Création du dossier de logs (au cas où)
mkdir -p slurm/log

echo "🚀 Job lancé sur $SLURMD_NODENAME"
nvidia-smi

# Lancement de ton script d'entraînement
srun python train.py

Writing launch.slurm


In [4]:
!sbatch launch.slurm

zsh:1: command not found: sbatch
